In [ ]:
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
import re
pattern = r"(\w+)_cells(\d+)_features(\d+)_([a-z1-9]+)_(\w+)_seed(\d+)_log"
optimal_df = pd.read_csv("../data/experiment_cellsim_broad/ground_truths_5000.csv")
skargs = dict(header=None, skipfooter=1, 
                      names=[
    "time(s)", "iter", "infeas", "ot_objective", "dual", "solver"
])
def load_df(path, **kwargs):
    dataset, ncells, nfeatures, metric, solver, seed = re.findall(pattern, path)[0]
    ncells, nfeatures, seed = map(int, [ncells, nfeatures, seed])
    
    if "sinkhorn" in path and "annealed" not in path:
        df = pd.read_csv(path, **skargs).iloc[1:]
    else:
        df = pd.read_csv(path, skipfooter=1)
    df['seed'] = seed; df['ncells'] = ncells; df["nfeatures"] = nfeatures; df['metric'] = metric
    df = df.merge(optimal_df, on=["metric", "seed", "nfeatures", "ncells"])
    df["compound_obj"] = np.abs(df["ot_objective"].astype(float) - df['cost']) + df["infeas"].astype(float) * df["Cinf"]
    df["time(s)"] = df["time(s)"].astype(float)
    return df
flist = glob("../data/experiment_cellsim_broad/*seed*_log.csv")
dflist = []
for f in tqdm(flist):
    df = load_df(f)
    dflist.append(df)
df = pd.concat(dflist).reset_index(drop=True)
arr = np.exp(pd.cut(np.log(df['time(s)']), bins=200 , precision=2,retbins=True)[0].apply(lambda x: x.mid).to_numpy())
# np.exp(arr[0].iloc[0].mid)
df['bintime'] = arr
df['Algorithm'] = df['solver'].replace({"lamp_kernel": "LAMP", 
                                        "annealed_sinkhorn_cellsim": "Anneal-SK",
                                        "sinkhorn_cellsim":  "SK"})
df["Metric"] = df['metric'].replace({
    "l1": "$\\ell_1$", 
    "l2": "$\\ell_2^2$",
    "correlation": "Pearson Correlation", 
    "cosine": "Cosine"
})
df.to_csv("../data_archive/fig_3_cellsim.csv", index=False)